# Exploración de la base de datos de Sputnik
Este cuaderno resume las tablas principales, ofrece estadística descriptiva e incluye visualizaciones para conocer la base de datos `sputnik.db`.

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.sparse import csr_matrix


sns.set_theme(style="whitegrid")
pd.options.display.float_format = "{:.2f}".format
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
DB_PATH = Path("../data/sputnik.db")
if not DB_PATH.exists():
    raise FileNotFoundError(f"No se encontró la base de datos en {DB_PATH.resolve()}")


def run_query(sql: str, params=None) -> pd.DataFrame:
    # Ejecuta consultas rápidas contra la base local y devuelve DataFrames listos para analizar.
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn, params=params)

## Inventario de tablas
Revisamos qué tablas existen y cuántos registros tiene cada una para priorizar el análisis.

In [ ]:
tables = run_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table' AND name NOT LIKE 'sqlite_%'
    ORDER BY name
""")

table_counts = []
for table_name in tables["name"]:
    row_count = run_query(f'SELECT COUNT(*) AS filas FROM "{table_name}"').iloc[0, 0]
    table_counts.append({"tabla": table_name, "filas": row_count})

table_counts_df = (
    pd.DataFrame(table_counts).sort_values("filas", ascending=False).reset_index(drop=True)
)

table_counts_df

In [ ]:
top_tables = table_counts_df.head(10)
plt.figure()
sns.barplot(data=top_tables, x="filas", y="tabla", palette="crest")
plt.title("Tablas con más registros")
plt.xlabel("Número de filas")
plt.ylabel("Tabla")
plt.tight_layout()

## Usuarios
Analizamos la actividad de los usuarios: volumen de calificaciones, soundoffs y roles declarados.

In [ ]:
users = run_query("""
    SELECT
        role,
        join_date,
        last_active,
        soundoffs,
        ratings_count,
        objectivity_score
    FROM users
""")

users["join_date"] = pd.to_datetime(users["join_date"], errors="coerce")
users["last_active"] = pd.to_datetime(users["last_active"], errors="coerce")

user_stats = users[["soundoffs", "ratings_count", "objectivity_score"]].describe().T
user_stats

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ratings = users["ratings_count"].dropna()
sns.histplot(ratings, ax=axes[0], bins=40, color="#4C72B0")
axes[0].set_title("Distribución de calificaciones por usuario")
axes[0].set_xlabel("ratings_count")
axes[0].set_ylabel("Usuarios")

role_counts = (
    users["role"]
    .fillna("desconocido")
    .value_counts()
    .reset_index(name="usuarios")
    .rename(columns={"index": "role"})
)
sns.barplot(data=role_counts, x="usuarios", y="role", ax=axes[1], palette="flare")
axes[1].set_title("Usuarios por rol")
axes[1].set_xlabel("Usuarios")
axes[1].set_ylabel("Rol")

plt.tight_layout()

## Lanzamientos
Exploramos métricas principales de los lanzamientos: tipo, año de publicación y desempeño en ratings.

In [ ]:
releases = run_query("""
    SELECT
        release_type,
        release_year,
        avg_rating,
        ratings_count,
        staff_avg,
        review_count
    FROM releases
""")

release_stats = releases[["avg_rating", "ratings_count", "staff_avg", "review_count"]].describe().T
release_stats

In [ ]:
plt.figure()
sns.countplot(
    data=releases,
    y="release_type",
    order=releases["release_type"].value_counts().index,
    palette="rocket",
)
plt.title("Lanzamientos por tipo")
plt.xlabel("Cantidad")
plt.ylabel("Tipo")
plt.tight_layout()

## Distribución del soporte de ratings
La mayoría de los lanzamientos tiene pocas calificaciones; analizamos la concentración del soporte para dimensionar la cola larga y ajustar modelos basados en popularidad.

In [ ]:
support = releases["ratings_count"].fillna(0).astype(int)
support_stats = (
    support.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).round(2).to_frame(name="valor")
)
display(support_stats)

thresholds = [1, 5, 10, 20, 50, 100, 200]
coverage = pd.DataFrame(
    {"umbral": thresholds, "porcentaje_releases": [(support <= t).mean() * 100 for t in thresholds]}
).round({"porcentaje_releases": 1})
display(coverage)

plt.figure(figsize=(10, 5))
sns.histplot(support, bins=50, log_scale=(True, False), color="#8C613C")
plt.axvline(support.median(), color="red", linestyle="--", linewidth=1, label="mediana")
plt.title("Distribución de ratings por lanzamiento")
plt.xlabel("ratings_count")
plt.ylabel("Lanzamientos")
plt.legend()
plt.tight_layout()

In [ ]:
year_counts = (
    releases.dropna(subset=["release_year"])
    .assign(release_year=lambda df: df["release_year"].astype(int))
    .groupby("release_year")
    .size()
    .reset_index(name="lanzamientos")
)

plt.figure(figsize=(12, 5))
sns.lineplot(data=year_counts, x="release_year", y="lanzamientos", marker="o", color="#1B998B")
plt.title("Evolución de lanzamientos por año")
plt.xlabel("Año")
plt.ylabel("Lanzamientos")
plt.tight_layout()

## Calificaciones e interacciones
Para evitar mover grandes volúmenes de datos, agregamos las interacciones directamente en SQL y graficamos las distribuciones resultantes.

In [ ]:
ratings_hist = run_query("""
    SELECT
        rating,
        COUNT(*) AS votos
    FROM interactions
    GROUP BY rating
    ORDER BY rating
""")

plt.figure()
sns.barplot(data=ratings_hist, x="rating", y="votos", palette="mako")
plt.title("Distribución de ratings")
plt.xlabel("Rating")
plt.ylabel("Votos")
plt.tight_layout()

In [ ]:
ratings_by_month = run_query("""
    SELECT
        substr(rating_date, 1, 7) AS mes,
        COUNT(*) AS votos
    FROM interactions
    WHERE rating_date IS NOT NULL AND rating_date <> ''
    GROUP BY mes
    ORDER BY mes
""")

ratings_by_month["mes"] = pd.to_datetime(ratings_by_month["mes"], format="%Y-%m", errors="coerce")
ratings_by_month = ratings_by_month.dropna(subset=["mes"])

plt.figure(figsize=(12, 5))
sns.lineplot(data=ratings_by_month, x="mes", y="votos", marker="o", color="#E4572E")
plt.title("Calificaciones registradas por mes")
plt.xlabel("Mes")
plt.ylabel("Votos")
plt.tight_layout()

## Interacciones: tamaño y densidad
Para un sistema de recomendación es clave entender la forma de la matriz usuario–ítem: cuántos usuarios, cuántos lanzamientos y cuán densa es (porcentaje de celdas con rating). Esto da una idea del sesgo hacia ítems populares y la viabilidad de enfoques basados en memoria.

In [ ]:
# Métricas básicas de la matriz usuario–ítem
n_users = run_query("SELECT COUNT(*) AS n FROM users").iloc[0, 0]
n_items = run_query("SELECT COUNT(*) AS n FROM releases").iloc[0, 0]
n_inter = run_query("SELECT COUNT(*) AS n FROM interactions").iloc[0, 0]

density = n_inter / (n_users * n_items) if (n_users and n_items) else 0.0
summary_df = pd.DataFrame(
    {
        "métrica": ["usuarios", "lanzamientos", "interacciones", "densidad"],
        "valor": [n_users, n_items, n_inter, density],
    }
)
summary_df

In [ ]:
# Muestra pequeña de heatmap (usuarios/ítems más activos)
# Nota: evitamos cargar todo el pivot si es grande; tomamos top-50 por actividad
# para una vista rápida.
top_users = run_query("""
    SELECT id_user, COUNT(*) AS c
    FROM interactions
    GROUP BY id_user
    ORDER BY c DESC
    LIMIT 50
""")
top_items = run_query("""
    SELECT id_release, COUNT(*) AS c
    FROM interactions
    GROUP BY id_release
    ORDER BY c DESC
    LIMIT 50
""")

parts = [
    "SELECT i.id_user, i.id_release, i.rating FROM interactions i",
    "JOIN (SELECT id_user FROM interactions GROUP BY id_user",
    "ORDER BY COUNT(*) DESC LIMIT 50) u USING(id_user)",
    "JOIN (SELECT id_release FROM interactions GROUP BY id_release",
    "ORDER BY COUNT(*) DESC LIMIT 50) r USING(id_release)",
]
sample_sql = " ".join(parts)
sample = run_query(sample_sql)

if not sample.empty:
    pivot = sample.pivot_table(
        index="id_user", columns="id_release", values="rating", aggfunc="mean"
    )
    plt.figure(figsize=(10, 8))
    sns.heatmap(pivot, cmap="YlGnBu", cbar_kws={"label": "rating"}, linewidths=0.1)
    plt.title("Heatmap de ratings (top-usuarios × top-ítems)")
    plt.xlabel("id_release")
    plt.ylabel("id_user")
    plt.tight_layout()
else:
    print("No hay suficientes datos para el heatmap de muestra.")

## Similitud entre lanzamientos (item–item)
Usamos una aproximación de filtrado colaborativo item–item con similitud coseno sobre ratings centrados por usuario. Limitamos a ítems con suficiente soporte para reducir ruido.

In [ ]:
# Parámetros de soporte para evitar ruido
MIN_USER_RATINGS = 5  # mínimo de calificaciones por usuario para incluirlo
MIN_ITEM_RATINGS = 20  # mínimo de calificaciones por lanzamiento
TOP_K_NEIGHBORS = 30  # vecinos por ítem a retener

# Extraemos interacciones filtradas y nombres de ítems
inter_df = run_query(f"""
    WITH user_ok AS (
        SELECT id_user
        FROM interactions
        GROUP BY id_user
        HAVING COUNT(*) >= {MIN_USER_RATINGS}
    ), item_ok AS (
        SELECT id_release
        FROM interactions
        GROUP BY id_release
        HAVING COUNT(*) >= {MIN_ITEM_RATINGS}
    )
    SELECT i.id_user, i.id_release, i.rating, r.title
    FROM interactions i
    JOIN user_ok u USING(id_user)
    JOIN item_ok s USING(id_release)
    JOIN releases r ON r.id_release = i.id_release
""")

if inter_df.empty:
    raise RuntimeError(
        "No hay suficientes datos tras el filtrado; ajusta MIN_USER_RATINGS/MIN_ITEM_RATINGS."
    )

# Centramos ratings por usuario (remueve sesgo del usuario)
user_means = inter_df.groupby("id_user")["rating"].mean()
inter_centered = inter_df.join(user_means, on="id_user", rsuffix="_mean")
inter_centered["rating_centered"] = inter_centered["rating"] - inter_centered["rating_mean"]

# Construimos matriz dispersa item × usuario con ratings centrados
item_cat = pd.Categorical(inter_centered["id_release"])
user_cat = pd.Categorical(inter_centered["id_user"])
rows = item_cat.codes.astype(np.int32)
cols = user_cat.codes.astype(np.int32)
data = inter_centered["rating_centered"].astype(np.float32).values

n_items = len(item_cat.categories)
n_users = len(user_cat.categories)
M = csr_matrix((data, (rows, cols)), shape=(n_items, n_users), dtype=np.float32)

# Intentamos usar vecinos más cercanos (cosine) para evitar construir toda la matriz de similitud
nn_inds = None
nn_sims = None
idx_to_item = np.array(item_cat.categories)  # mapa índice->id_release
id_to_idx = {int(idx_to_item[i]): i for i in range(n_items)}

try:
    from sklearn.neighbors import NearestNeighbors  # noqa: E402

    n_neighbors = min(
        TOP_K_NEIGHBORS + 1, max(2, n_items)
    )  # +1 para incluir self y luego excluirlo
    nn = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=n_neighbors)
    nn.fit(M)
    dists, inds = nn.kneighbors(M, return_distance=True)
    sims = 1.0 - dists
    # Excluimos el self en la posición 0 si corresponde
    if inds.shape[1] > 0 and np.all(inds[:, 0] == np.arange(n_items)):
        inds = inds[:, 1:]
        sims = sims[:, 1:]
    nn_inds, nn_sims = inds, sims
    sim_items = None  # no construimos la matriz completa para ahorrar memoria
except Exception:
    # Fallback denso y acotado si sklearn no está disponible
    MAX_ITEMS_DENSE = 1500
    if n_items > MAX_ITEMS_DENSE:
        # Tomamos los ítems más populares para la aproximación densa
        top_items = inter_df["id_release"].value_counts().head(MAX_ITEMS_DENSE).index.tolist()
        inter_small = inter_centered[inter_centered["id_release"].isin(top_items)].copy()
        item_cat2 = pd.Categorical(inter_small["id_release"])
        user_cat2 = pd.Categorical(inter_small["id_user"])
        rows2 = item_cat2.codes.astype(np.int32)
        cols2 = user_cat2.codes.astype(np.int32)
        data2 = inter_small["rating_centered"].astype(np.float32).values
        M2 = csr_matrix(
            (data2, (rows2, cols2)),
            shape=(len(item_cat2.categories), len(user_cat2.categories)),
            dtype=np.float32,
        )
        X = M2.toarray().astype(np.float32)
        norms = np.linalg.norm(X, axis=1, keepdims=True) + 1e-9
        Xn = X / norms
        S = (Xn @ Xn.T).astype(np.float32)
        sim_items = pd.DataFrame(S, index=item_cat2.categories, columns=item_cat2.categories)
        nn_inds = nn_sims = None
        idx_to_item = np.array(item_cat2.categories)
        id_to_idx = {int(idx_to_item[i]): i for i in range(len(idx_to_item))}
    else:
        # Denso directo si el tamaño es manejable
        X = M.toarray().astype(np.float32)
        norms = np.linalg.norm(X, axis=1, keepdims=True) + 1e-9
        Xn = X / norms
        S = (Xn @ Xn.T).astype(np.float32)
        sim_items = pd.DataFrame(S, index=item_cat.categories, columns=item_cat.categories)
        nn_inds = nn_sims = None

# Reporte de forma útil para depurar
print(
    {
        "n_items": n_items,
        "n_users": n_users,
        "mode": "NN" if nn_inds is not None else "dense",
        "TOP_K": TOP_K_NEIGHBORS,
    }
)

In [ ]:
# Helper para consultar similares a un release dado
def similares_a(id_release: int, k: int = 10) -> pd.DataFrame:
    # Si tenemos estructura de vecinos, usamos esa (más eficiente)
    if "nn_inds" in globals() and nn_inds is not None:
        if id_release not in id_to_idx:
            raise KeyError("El release no está en la matriz (quizá por filtros de soporte)")
        i = id_to_idx[id_release]
        top_k = min(k, nn_inds.shape[1])
        idxs = nn_inds[i, :top_k]
        sims = nn_sims[i, :top_k]
        ids = idx_to_item[idxs]
        ids_str = ",".join(map(str, ids.tolist()))
        meta = run_query(
            "SELECT id_release, title FROM releases WHERE id_release IN (%s)" % ids_str
        )
        out = meta.set_index("id_release").loc[ids].reset_index()
        out["similaridad"] = sims
        return out
    # Si no, caemos a la matriz completa sim_items
    if "sim_items" not in globals() or sim_items is None or id_release not in sim_items.index:
        raise KeyError("El release no está en la matriz (quizá por filtros de soporte)")
    sims = sim_items.loc[id_release].drop(index=id_release).sort_values(ascending=False).head(k)
    ids_str = ",".join(map(str, sims.index.tolist()))
    meta = run_query("SELECT id_release, title FROM releases WHERE id_release IN (%s)" % ids_str)
    out = meta.set_index("id_release").loc[sims.index].reset_index()
    out["similaridad"] = sims.values
    return out


# Ejemplo: tomamos un ítem popular y listamos sus 10 similares
ej_popular = inter_df["id_release"].value_counts().index[0]
similares_a(ej_popular, k=10)

## Similitud entre usuarios (user–user)
Buscamos vecinos cercanos de un usuario en el espacio de ratings centrados. Esto ayuda a inspeccionar calidad de vecindarios y posibles problemas de sparsidad.

In [ ]:
# Reutilizamos inter_df pero cambiamos la vista a usuario × ítem centrado
user_means2 = inter_df.groupby("id_user")["rating"].mean()
df2 = inter_df.join(user_means2, on="id_user", rsuffix="_mean")
df2["rating_centered"] = df2["rating"] - df2["rating_mean"]

# Matriz dispersa usuario × ítem
u_cat = pd.Categorical(df2["id_user"])
i_cat = pd.Categorical(df2["id_release"])
rows = u_cat.codes.astype(np.int32)
cols = i_cat.codes.astype(np.int32)
data = df2["rating_centered"].astype(np.float32).values
U = csr_matrix(
    (data, (rows, cols)), shape=(len(u_cat.categories), len(i_cat.categories)), dtype=np.float32
)

# Vecinos de usuarios con NearestNeighbors (cosine)
inds_u = None
sims_u = None
sim_users = None
try:
    from sklearn.neighbors import NearestNeighbors  # noqa: E402

    n_users_mat = U.shape[0]
    n_neighbors = min(31, max(2, n_users_mat))
    nn_u = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=n_neighbors)
    nn_u.fit(U)
    dists_u, inds_u = nn_u.kneighbors(U, return_distance=True)
    sims_u = 1.0 - dists_u
    # remover self si está en primera posición
    if inds_u.shape[1] > 0 and np.all(inds_u[:, 0] == np.arange(n_users_mat)):
        inds_u = inds_u[:, 1:]
        sims_u = sims_u[:, 1:]
    idx_to_user = np.array(u_cat.categories)
    user_to_idx = {idx_to_user[i]: i for i in range(n_users_mat)}
except Exception:
    # Fallback denso acotado
    MAX_USERS_DENSE = 3000
    if U.shape[0] > MAX_USERS_DENSE:
        sample_users = inter_df["id_user"].value_counts().head(MAX_USERS_DENSE).index.tolist()
        df_small = df2[df2["id_user"].isin(sample_users)].copy()
        u_cat2 = pd.Categorical(df_small["id_user"])
        i_cat2 = pd.Categorical(df_small["id_release"])
        rows2 = u_cat2.codes.astype(np.int32)
        cols2 = i_cat2.codes.astype(np.int32)
        data2 = df_small["rating_centered"].astype(np.float32).values
        U2 = csr_matrix(
            (data2, (rows2, cols2)),
            shape=(len(u_cat2.categories), len(i_cat2.categories)),
            dtype=np.float32,
        )
        X = U2.toarray().astype(np.float32)
        norms = np.linalg.norm(X, axis=1, keepdims=True) + 1e-9
        Xn = X / norms
        S = (Xn @ Xn.T).astype(np.float32)
        sim_users = pd.DataFrame(S, index=u_cat2.categories, columns=u_cat2.categories)
        idx_to_user = np.array(u_cat2.categories)
        user_to_idx = {idx_to_user[i]: i for i in range(len(idx_to_user))}
    else:
        X = U.toarray().astype(np.float32)
        norms = np.linalg.norm(X, axis=1, keepdims=True) + 1e-9
        Xn = X / norms
        S = (Xn @ Xn.T).astype(np.float32)
        sim_users = pd.DataFrame(S, index=u_cat.categories, columns=u_cat.categories)
        idx_to_user = np.array(u_cat.categories)
        user_to_idx = {idx_to_user[i]: i for i in range(len(idx_to_user))}

print(
    {"n_users": U.shape[0], "n_items": U.shape[1], "mode": "NN" if inds_u is not None else "dense"}
)

In [ ]:
# Helper: vecinos más similares a un usuario dado
def vecinos_de(user_id, k: int = 10) -> pd.DataFrame:
    if "user_to_idx" not in globals():
        raise RuntimeError("Ejecutá la celda de similitud de usuarios antes de consultar vecinos.")

    lookup_id = user_id
    if user_to_idx:
        sample_key = next(iter(user_to_idx))
        if lookup_id not in user_to_idx:
            try:
                lookup_id = type(sample_key)(user_id)
            except Exception:
                pass

    if "inds_u" in globals() and inds_u is not None and lookup_id in user_to_idx:
        u = user_to_idx[lookup_id]
        top_k = min(k, inds_u.shape[1])
        idxs = inds_u[u, :top_k]
        sims = sims_u[u, :top_k]
        ids = idx_to_user[idxs]
        return pd.DataFrame({"id_user": ids, "similaridad": sims})

    if sim_users is None:
        raise KeyError("El usuario no está en la matriz (quizá por filtros de soporte)")

    if lookup_id not in sim_users.index:
        try:
            lookup_id = sim_users.index.dtype.type(user_id)
        except Exception:
            pass

    if lookup_id not in sim_users.index:
        raise KeyError("El usuario no está en la matriz (quizá por filtros de soporte)")

    sims = sim_users.loc[lookup_id].drop(index=lookup_id).sort_values(ascending=False).head(k)
    return pd.DataFrame({"id_user": sims.index, "similaridad": sims.values})


# Ejemplo con un usuario activo
ej_user = inter_df["id_user"].value_counts().index[0]
vecinos_de(ej_user, k=10)

## Géneros: soporte y calidad promedio
Analizamos el promedio de rating por género y su soporte mínimo para evitar outliers. También mostramos los géneros más frecuentes.

In [ ]:
MIN_SUPPORT_GENRE = 100
genre_stats = run_query("""
    SELECT g.name AS genre, COUNT(*) AS n, AVG(i.rating) AS mean_rating
    FROM interactions i
    JOIN releases r ON r.id_release = i.id_release
    JOIN release_genres rg ON rg.id_release = r.id_release
    JOIN genres g ON g.id_genre = rg.id_genre
    GROUP BY g.name
""")

top_genres = genre_stats.sort_values("n", ascending=False).head(20)
plt.figure(figsize=(10, 6))
sns.barplot(data=top_genres, x="n", y="genre", palette="viridis")
plt.title("Géneros más frecuentes (top 20)")
plt.xlabel("Interacciones")
plt.ylabel("")
plt.tight_layout()

strong_genres = (
    genre_stats.query("n >= @MIN_SUPPORT_GENRE")
    .sort_values("mean_rating", ascending=False)
    .head(20)
)
plt.figure(figsize=(10, 6))
sns.barplot(data=strong_genres, x="mean_rating", y="genre", palette="mako")
plt.title("Mejor rating medio por género (soporte ≥ %d)" % MIN_SUPPORT_GENRE)
plt.xlabel("Rating medio")
plt.ylabel("")
plt.tight_layout()